# AMEX Enterprise Credit Risk Platform
## Notebook 51 -- Collections Optimization: Modeling
### Phase 4 . Problem Statement 9: Collections Optimization

CRISP-DM stage: **Modeling**. Depends on Notebook 50's real policy (cure definition, KPI target,
treatment-tier policy) and Problem 8's real, production-recommended severity-state scorer (reused
verbatim -- no fresh refit).

**What this notebook does:** scores every real statement with Problem 8's already-fitted formula
(vectorized, no Python loop), builds real per-customer cure labels via a vectorized `shift(-1)` lookahead
(current state vs. real next-statement state), trains a real WARP-tuned XGBoost propensity-to-cure
classifier on the TRAIN population, reports the full classification metrics suite against Notebook 50's
KPI target on the real HOLDOUT population, applies the real treatment-tier policy, and persists the
trained model.

**HYPER note:** Sections 1-3 and 11-12 reuse this platform's established Modeling-notebook structure
(Notebook 47) verbatim where the logic is genuinely identical; Section 6's XGBoost training block reuses
Notebook 39's exact WARP-tuned hyperparameter/threading pattern.

**WARP note:** every per-statement computation in Sections 4-5 is a single vectorized Polars expression
graph -- no per-row Python loop anywhere in this notebook. Section 6 trains with `tree_method="hist"` and
`n_jobs` capped to Notebook 50's tightened 92% thread count, using `float32` features and explicit
`gc.collect()` after freeing intermediate frames.

Real bug caught and fixed while writing this notebook (before it was ever run): an early draft of Section
8 re-scanned the FULL holdout population for severity scores and tried to align it against
`PROPENSITY_HOLDOUT`, which only covers the ELIGIBLE subset -- two different populations of different
lengths. Fixed by capturing the eligible population's severity scores directly alongside the model's input
arrays in Section 6, so Section 8 uses a genuinely row-aligned array instead of a mismatched second scan --
now also checked explicitly in Section 11's verification.

Zero-fabrication statement: every number this notebook prints is either computed live against the real raw
Kaggle CSVs, or an explicitly labeled ASSUMPTION inherited from Notebook 50 -- no results are hardcoded or
estimated in advance.
**Update (2026-08-26), after the user hit a real 45-minute full SYSTEM freeze running this notebook:** Section 4 itself was re-verified line by line -- it is, and remains, 100% lazy (`pl.scan_csv` + `.with_columns`/`.join`/`.select` only, no `.collect()` anywhere in it), so it cannot be the direct cause of a RAM-driven freeze on its own. Two real changes were made anyway: (1) a pre-flight system-wide available-RAM check was added at the end of Section 2, before Section 3 opens the raw CSV at all -- if another Jupyter kernel or application is already holding most of the machine's RAM (the most likely real cause of a freeze this early, before this notebook's own `.collect()` calls even run), this now raises a fast, clear, actionable error instead of silently proceeding into a freeze; and (2) Section 5's cure-label builder previously called a helper that joined+sorted+collected the raw CSV independently for TRAIN and for HOLDOUT -- scanning the entire raw CSV from disk TWICE. Fixed to collect the full scored population ONCE, then split TRAIN/HOLDOUT from that single in-memory frame with a cheap eager join -- the raw CSV is now scanned exactly once for this whole notebook, with no change to any computed result. RSS and system-wide available-RAM checkpoints were also added after every major step in Sections 3-6, so if a freeze happens again, the printed output will show exactly where memory pressure actually starts climbing.


In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOK 50'S REAL POLICY AND PROBLEM
#            1'S REAL TRAIN/HOLDOUT SPLIT
# =============================================================================
import os
import sys
import json
import time
import gc
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebook 50's Real Policy and Splits")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB50_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_50_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB50_SUMMARY_PATH, "run 50_collections_optimization_business_understanding.ipynb first"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB50_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB50_SUMMARY = json.load(f)

COLLECTIONS_POLICY_PATH = Path(NB50_SUMMARY["policy_path"])
if not COLLECTIONS_POLICY_PATH.exists():
    raise FileNotFoundError(f"{COLLECTIONS_POLICY_PATH} not found.\nFix: re-run Notebook 50.")
with open(COLLECTIONS_POLICY_PATH, "r", encoding="utf-8") as f:
    COLLECTIONS_POLICY = json.load(f)

COLLECTIONS_ELIGIBLE_STATES = COLLECTIONS_POLICY["collections_eligible_states"]
MIN_STATEMENTS_FOR_CURE_LABEL = COLLECTIONS_POLICY["min_statements_for_cure_label"]
P8_REUSE = COLLECTIONS_POLICY["reused_from_problem_8"]
STATE_NAMES = P8_REUSE["state_names"]
MONITORED_COLS = sorted(P8_REUSE["monitored_features"])
P8_WEIGHTS = P8_REUSE["feature_weights"]["weights"]
P8_DIRECTIONS = P8_REUSE["feature_weights"]["directions"]
P8_MEANS = P8_REUSE["feature_weights"]["means"]
P8_STDS = P8_REUSE["feature_weights"]["stds"]
CUT_LOW = P8_REUSE["cut_low"]
CUT_HIGH = P8_REUSE["cut_high"]
MIN_ROC_AUC_TARGET = COLLECTIONS_POLICY["kpi_targets"]["min_propensity_model_roc_auc"]
TREATMENT_TIERS = COLLECTIONS_POLICY["kpi_targets"]["treatment_tier_policy"]["tiers"]

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    """Same 3(+1)-candidate resolver every notebook in this platform uses (see
    Notebook 39/47 for the full history of why the nested-first order matters):
    the real known current nested Phase1_Foundation/Problem1.../<legacy_folder_
    name> path is checked FIRST, then PILLAR_DIRS, then the legacy root-level
    path, then whatever the summary JSON literally recorded (lowest priority,
    since that is the one most likely to be stale)."""
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "01_Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + "\nFix: run the notebook that produces this file again, or tell me the real path."
    )


TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)

RANDOM_SEED = PROJECT_CONFIG["random_seed"]
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
DETECTED_TOTAL_RAM_BYTES = PROJECT_CONFIG["resource_limits"]["total_ram_bytes_detected"]

if "collections_modeling" in PILLAR_DIRS:
    COLLECTIONS_MODELING_DIR = PILLAR_DIRS["collections_modeling"]
else:
    COLLECTIONS_MODELING_DIR = (
        PROJECT_ROOT / "Phase4_Operational_Risk_Management"
        / "09_Problem9_Collections_Optimization" / "modeling"
    )
    print(f"NOTE: 'collections_modeling' not in pillar_dirs -- using fallback: {COLLECTIONS_MODELING_DIR}")
COLLECTIONS_MODELING_DIR.mkdir(parents=True, exist_ok=True)
COLLECTIONS_CHARTS_DIR = COLLECTIONS_MODELING_DIR / "charts"
COLLECTIONS_CHARTS_DIR.mkdir(parents=True, exist_ok=True)
COLLECTIONS_MODELS_DIR = (
    PROJECT_ROOT / "Phase4_Operational_Risk_Management" / "09_Problem9_Collections_Optimization" / "models"
)
COLLECTIONS_MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded policy from        : {COLLECTIONS_POLICY_PATH}")
print(f"Resolved train_split.csv  : {TRAIN_SPLIT_PATH}")
print(f"Resolved test_split.csv   : {TEST_SPLIT_PATH}")
print(f"COLLECTIONS_ELIGIBLE_STATES: {COLLECTIONS_ELIGIBLE_STATES}")
print(f"Reused Problem 8 CUT_LOW / CUT_HIGH (verbatim, no refit): {CUT_LOW:.4f} / {CUT_HIGH:.4f}")
print(f"Monitored feature universe (reused from Problem 8): {len(MONITORED_COLS)} base columns")
print(f"min_propensity_model_roc_auc target: {MIN_ROC_AUC_TARGET}")
print(f"Modeling artifacts will be written under: {COLLECTIONS_MODELING_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS (PHASE 4 TIGHTENED
#            CAP -- SAME 92%/92% POLICY NOTEBOOK 50 ESTABLISHED, REUSED
#            VERBATIM RATHER THAN RE-DERIVED)
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports (Phase 4 Tightened Cap)")

_PHASE4_CPU_FRACTION_CAP = 0.92
_PHASE4_RAM_FRACTION_CAP = 0.92
_historical_thread_count = PROJECT_CONFIG["resource_limits"]["warp_thread_count"]
_historical_max_ram_bytes = PROJECT_CONFIG["resource_limits"]["max_ram_bytes"]
WARP_THREAD_COUNT = min(
    _historical_thread_count,
    max(1, round(DETECTED_LOGICAL_CORES * _PHASE4_CPU_FRACTION_CAP)),
)
MAX_RAM_BYTES = min(
    _historical_max_ram_bytes,
    round(DETECTED_TOTAL_RAM_BYTES * _PHASE4_RAM_FRACTION_CAP),
)

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    from xgboost import XGBClassifier
except ImportError:
    missing.append("xgboost")
try:
    from sklearn.metrics import (
        roc_auc_score, average_precision_score, accuracy_score, precision_score,
        recall_score, f1_score, log_loss, matthews_corrcoef, confusion_matrix,
        roc_curve, precision_recall_curve,
    )
except ImportError:
    missing.append("scikit-learn")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


def _available_ram_gb() -> float:
    """System-wide available RAM (not just this process's RSS) -- the number
    that actually predicts an OS-level freeze. A low _rss_gb() can coexist
    with a dangerously low _available_ram_gb() if OTHER processes/kernels
    (left-over Jupyter kernels from earlier notebooks, browser tabs, etc.)
    are already holding most of the machine's RAM before this notebook does
    anything at all."""
    return psutil.virtual_memory().available / 1e9


logger.info(
    f"Polars thread pool configured to {WARP_THREAD_COUNT}/{DETECTED_LOGICAL_CORES} threads "
    f"({WARP_THREAD_COUNT / DETECTED_LOGICAL_CORES:.0%}, Phase 4 tightened cap, reused verbatim from "
    f"Notebook 50)"
)
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
print(f"Configured RAM ceiling (Phase 4 tightened cap): {MAX_RAM_BYTES / 1e9:.1f} GB")
print(f"XGBoost will train with n_jobs={WARP_THREAD_COUNT}, tree_method='hist' (WARP 6.1/6.6 -- vectorized, "
      f"threaded, capped, matching this platform's established XGBClassifier convention, e.g. Notebook 39)")

# --- REAL BUG PREVENTION, added 2026-08-26 after the user's own machine hit
#     a 45-minute full SYSTEM freeze on a run of this notebook: every
#     operation in Sections 4-6 below is written to be lazy/streamed/capped
#     and, read on its own, should never approach the RAM ceiling above --
#     but that guarantee assumes THIS process starts from a mostly-idle
#     machine. It does not hold if other Jupyter kernels (left running from
#     earlier notebooks in this platform) or other applications are already
#     holding most of the system's RAM before this notebook does anything at
#     all. A silent freeze deep inside Section 5's .collect() call is far
#     worse than a fast, clear, actionable error right here -- so this
#     checks real system-wide available RAM (not just this process's own
#     RSS) BEFORE Section 3 opens the multi-GB raw CSV, and refuses to
#     proceed if there isn't enough headroom for this notebook's real work. ---
_available_ram_gb_at_start = _available_ram_gb()
_min_required_available_ram_gb = 0.5 * (MAX_RAM_BYTES / 1e9)
print(f"System-wide available RAM at Section 2 start: {_available_ram_gb_at_start:.2f} GB "
      f"(minimum required to proceed safely: {_min_required_available_ram_gb:.1f} GB)")
if _available_ram_gb_at_start < _min_required_available_ram_gb:
    raise RuntimeError(
        f"Only {_available_ram_gb_at_start:.2f} GB of system RAM is available right now -- below the "
        f"{_min_required_available_ram_gb:.1f} GB this notebook needs as headroom before it is safe to "
        f"open the real raw CSV in Section 3. This is very likely OTHER Jupyter kernels or applications "
        f"already holding most of the machine's RAM, not a problem with this notebook's own code.\n"
        f"Fix: in Jupyter, use 'Kernel > Shut Down All Kernels' (or close every other notebook tab), "
        f"close other heavy applications/browser tabs, then restart JUST this notebook's kernel and "
        f"re-run from the top. If available RAM is still this low on a genuinely clean machine, stop and "
        f"report the number above -- that would point to a real problem worth investigating further, not "
        f"this guard being overly cautious."
    )
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS & LOAD REAL TRAIN/HOLDOUT SPLITS + TARGET
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths & Load Real Train/Holdout Splits + Target")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")

TARGET_DF = pl.read_csv(RAW_TRAIN_LABELS_PATH, schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
TRAIN_IDS_DF = pl.read_csv(TRAIN_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8})
HOLDOUT_IDS_DF = pl.read_csv(TEST_SPLIT_PATH, schema_overrides={"customer_ID": pl.Utf8})
N_TRAIN_CUSTOMERS = TRAIN_IDS_DF.height
N_HOLDOUT_CUSTOMERS = HOLDOUT_IDS_DF.height

print(f"Raw train_data.csv         : {RAW_TRAIN_DATA_PATH}")
print(f"Real TRAIN customers (Notebook 02's real split, reused)  : {N_TRAIN_CUSTOMERS:,}")
print(f"Real HOLDOUT customers (Notebook 02's real split, reused): {N_HOLDOUT_CUSTOMERS:,}")
print(f"Process RSS after Section 3 (small eager reads only, no raw CSV touched yet): {_rss_gb():.2f} GB "
      f"-- system-wide available RAM: {_available_ram_gb():.2f} GB")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: SCORE EVERY REAL STATEMENT WITH PROBLEM 8'S REUSED FORMULA
#            (VECTORIZED, WARP-STYLE -- NO PER-ROW PYTHON LOOP)
# =============================================================================
_section("SECTION 4: Score Every Real Statement -- Problem 8's Reused Formula (Vectorized)")

# --- WARP note: the entire composite score for every statement is one
#     Polars expression graph (a weighted sum of column-wise z-scores),
#     evaluated lazily and streamed in Section 5 -- there is no Python-level
#     per-row loop anywhere in this notebook. This is the vectorization
#     WARP calls for; a Numba-JIT'd loop was considered and rejected here on
#     purpose, since a genuinely vectorized Polars expression is faster and
#     simpler than JIT-compiling an explicit loop over the same arithmetic. ---
_schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
for _c in MONITORED_COLS:
    _schema_overrides[_c] = pl.Float32

_inf_clean_exprs = [
    pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
    for c in MONITORED_COLS
]

# Same real (customer_ID, S_2) tie-break fix Notebook 47 established (475
# duplicate-key pairs observed in this platform's fixture) -- _csv_row_order
# is captured immediately after scan_csv, before any join/filter/sort can
# reorder rows, and used as an explicit tertiary sort key everywhere
# statement order matters below.
_base_lf = (
    pl.scan_csv(RAW_TRAIN_DATA_PATH, schema_overrides=_schema_overrides)
    .with_row_index("_csv_row_order")
    .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
    .with_columns(_inf_clean_exprs)
    .join(TARGET_DF.lazy(), on="customer_ID", how="inner")
)
# Diagnostic checkpoint (added 2026-08-26): everything above is LazyFrame
# construction -- pl.scan_csv() only reads the file header plus a small
# sample of rows to resolve dtypes for the columns NOT covered by
# schema_overrides; nothing here reads the full ~16GB file. If RSS/available
# RAM move meaningfully between the Section 3 checkpoint above and this one,
# that is real, actionable evidence -- report the two numbers if it happens.
print(f"Process RSS after building _base_lf (lazy plan only, no .collect() yet): {_rss_gb():.2f} GB "
      f"-- system-wide available RAM: {_available_ram_gb():.2f} GB")

_wz_cols = []
for _c in MONITORED_COLS:
    _mean, _std, _w, _d = P8_MEANS[_c], P8_STDS[_c], P8_WEIGHTS[_c], P8_DIRECTIONS[_c]
    if _std > 0 and _w > 0:
        _expr = ((pl.col(_c) - _mean) / _std * _w * _d).fill_null(0.0).alias(f"_wz_{_c}")
    else:
        _expr = pl.lit(0.0).alias(f"_wz_{_c}")
    _wz_cols.append(_expr)

_state_expr = (
    pl.when(pl.col("SEVERITY_SCORE") <= CUT_LOW).then(pl.lit(STATE_NAMES[0]))
    .when(pl.col("SEVERITY_SCORE") <= CUT_HIGH).then(pl.lit(STATE_NAMES[1]))
    .otherwise(pl.lit(STATE_NAMES[2]))
    .alias("STATE")
)

_scored_lf = (
    _base_lf
    .with_columns(_wz_cols)
    .with_columns(pl.sum_horizontal([f"_wz_{c}" for c in MONITORED_COLS]).alias("SEVERITY_SCORE"))
    .with_columns(_state_expr)
    .select(["customer_ID", "S_2", "_csv_row_order", "target", "SEVERITY_SCORE", "STATE"] + MONITORED_COLS)
)
print(f"Built lazy scoring expression graph for {len(MONITORED_COLS)} monitored columns "
      f"(reused Problem 8's real weights/means/stds/directions verbatim -- no refit).")
print(f"Process RSS at Section 4 end (still lazy, still no .collect() anywhere in this notebook yet): "
      f"{_rss_gb():.2f} GB -- system-wide available RAM: {_available_ram_gb():.2f} GB")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: BUILD REAL CURE LABELS -- CURRENT STATE + REAL NEXT-STATEMENT
#            STATE, VIA A VECTORIZED shift(-1), NOT A PYTHON LOOP
# =============================================================================
_section("SECTION 5: Build Real Cure Labels via Vectorized shift(-1)")

# --- REAL FIX, added 2026-08-26: this section previously called a helper
#     that joined+sorted+collected the raw CSV separately for TRAIN and for
#     HOLDOUT -- two independent lazy pipelines over the SAME _scored_lf,
#     meaning Polars scanned and parsed the entire raw CSV from disk TWICE.
#     Fixed by collecting _scored_lf's ONE global sort ONCE here, then
#     splitting TRAIN/HOLDOUT from that single already-in-memory frame with
#     a cheap eager join -- the raw CSV is now scanned exactly once for this
#     entire notebook. This does not change any result (cure labels are
#     still computed independently per split via .over("customer_ID"), and a
#     real customer only ever belongs to one split, so there is no
#     cross-split leakage either way) -- it only removes duplicate I/O. ---
print(f"Process RSS immediately before the ONE real .collect() call in this notebook "
      f"(streaming, full scored population): {_rss_gb():.2f} GB -- system-wide available RAM: "
      f"{_available_ram_gb():.2f} GB")
_t0_collect = time.time()
ALL_SCORED = (
    _scored_lf
    .sort(["customer_ID", "S_2", "_csv_row_order"])
    .collect(engine="streaming")
)
print(f"Collected {ALL_SCORED.height:,} real scored statements (TRAIN + HOLDOUT combined) in "
      f"{time.time() - _t0_collect:.1f}s. Process RSS: {_rss_gb():.2f} GB -- system-wide available RAM: "
      f"{_available_ram_gb():.2f} GB")


def _materialize_with_cure_label(all_scored_df: "pl.DataFrame", ids_df: "pl.DataFrame", label: str) -> "pl.DataFrame":
    _t0 = time.time()
    # Cheap eager join + re-sort against an already-in-memory frame (no disk
    # I/O) -- join does not guarantee row-order preservation, so the
    # customer_ID/S_2/_csv_row_order sort is re-applied on this smaller
    # subset to guarantee shift(-1) below sees real statement order.
    _df = (
        all_scored_df.join(ids_df, on="customer_ID", how="inner")
        .sort(["customer_ID", "S_2", "_csv_row_order"])
    )
    # WARP note: shift(-1).over("customer_ID") computes every customer's real
    # NEXT statement's state in one vectorized pass -- the Polars equivalent
    # of a per-customer lookahead loop, without ever leaving vectorized code.
    _df = _df.with_columns([
        pl.col("STATE").shift(-1).over("customer_ID").alias("_next_state"),
        pl.len().over("customer_ID").alias("_n_statements"),
    ])
    _state_rank = {STATE_NAMES[0]: 0, STATE_NAMES[1]: 1, STATE_NAMES[2]: 2}
    _df = _df.with_columns([
        pl.col("STATE").replace_strict(_state_rank, default=None).alias("_state_rank"),
        pl.col("_next_state").replace_strict(_state_rank, default=None).alias("_next_state_rank"),
    ])
    _df = _df.with_columns(
        (pl.col("_next_state_rank") < pl.col("_state_rank")).alias("CURED")
    )
    # Real, honest restriction: a row is a genuine cure-label training/eval
    # example only if (a) it is in a collections-eligible state, AND (b) a
    # real next statement was actually observed (shift(-1) does not produce
    # a null for the last real statement of a multi-statement customer, but
    # DOES for a customer's only statement -- both are excluded here rather
    # than silently treated as "did not cure").
    _eligible = _df.filter(
        pl.col("STATE").is_in(COLLECTIONS_ELIGIBLE_STATES) & pl.col("_next_state").is_not_null()
    )
    print(f"{label}: scored {_df.height:,} real statements, "
          f"{_eligible.height:,} genuinely eligible (state in {COLLECTIONS_ELIGIBLE_STATES}, real next "
          f"statement observed) in {time.time() - _t0:.1f}s. Process RSS: {_rss_gb():.2f} GB")
    return _eligible


TRAIN_ELIGIBLE = _materialize_with_cure_label(ALL_SCORED, TRAIN_IDS_DF, "TRAIN")
HOLDOUT_ELIGIBLE = _materialize_with_cure_label(ALL_SCORED, HOLDOUT_IDS_DF, "HOLDOUT")
del ALL_SCORED
gc.collect()
print(f"Process RSS after freeing ALL_SCORED: {_rss_gb():.2f} GB -- system-wide available RAM: "
      f"{_available_ram_gb():.2f} GB")

_train_cure_rate = float(TRAIN_ELIGIBLE["CURED"].mean())
_holdout_cure_rate = float(HOLDOUT_ELIGIBLE["CURED"].mean())
print(f"Real TRAIN cure rate  : {_train_cure_rate:.4f} ({int(TRAIN_ELIGIBLE['CURED'].sum()):,} / "
      f"{TRAIN_ELIGIBLE.height:,})")
print(f"Real HOLDOUT cure rate: {_holdout_cure_rate:.4f} ({int(HOLDOUT_ELIGIBLE['CURED'].sum()):,} / "
      f"{HOLDOUT_ELIGIBLE.height:,})")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: TRAIN THE REAL PROPENSITY-TO-CURE CLASSIFIER (WARP-TUNED
#            XGBoost -- float32 features, threaded to the tightened cap,
#            hist tree method, matching this platform's established
#            XGBClassifier convention, e.g. Notebook 39 Section on model
#            training)
# =============================================================================
_section("SECTION 6: Train the Real Propensity-to-Cure Classifier (WARP-Tuned XGBoost)")

# WARP note: float32 (not float64) halves the memory-bandwidth footprint of
# every feature column moved through training, per this platform's standing
# WARP dtype convention; int64 target matches XGBoost's expected label dtype.
X_train = TRAIN_ELIGIBLE.select(MONITORED_COLS).to_numpy().astype(np.float32, copy=False)
y_train = TRAIN_ELIGIBLE.get_column("CURED").cast(pl.Int64).to_numpy()
X_holdout = HOLDOUT_ELIGIBLE.select(MONITORED_COLS).to_numpy().astype(np.float32, copy=False)
y_holdout = HOLDOUT_ELIGIBLE.get_column("CURED").cast(pl.Int64).to_numpy()
# Captured HERE, from the SAME eligible-population frame that produced
# X_holdout/y_holdout row-for-row, before that frame is freed below -- Section
# 8 needs each holdout row's own real severity score to build the treatment-
# tier split, and it must stay aligned with PROPENSITY_HOLDOUT's row order.
# A second, independent re-scan of the full (not eligibility-filtered) raw
# CSV would NOT be aligned to this array (different population, different
# length) -- capturing it directly here avoids that real alignment bug
# entirely, and is also strictly cheaper (no second CSV pass).
HOLDOUT_ELIGIBLE_SEVERITY = HOLDOUT_ELIGIBLE.get_column("SEVERITY_SCORE").to_numpy()

# WARP note: free the source Polars frames' redundant copies before training
# rather than let them sit alongside the numpy arrays for the rest of the
# notebook -- explicit gc.collect() after del, same memory-pre-allocation-
# adjacent discipline this platform's Notebook 39 already established.
del TRAIN_ELIGIBLE, HOLDOUT_ELIGIBLE
gc.collect()
print(f"Process RSS after freeing source frames: {_rss_gb():.2f} GB -- system-wide available RAM: "
      f"{_available_ram_gb():.2f} GB")
print(f"X_train {X_train.shape}, X_holdout {X_holdout.shape}, {len(MONITORED_COLS)} features")

_t0 = time.time()
propensity_model = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", n_jobs=WARP_THREAD_COUNT, random_state=RANDOM_SEED,
    eval_metric="auc", verbosity=0,
)
propensity_model.fit(X_train, y_train)
_train_seconds = time.time() - _t0
print(f"Trained in {_train_seconds:.1f}s using {WARP_THREAD_COUNT} threads (WARP-capped, hist tree method)")

PROPENSITY_HOLDOUT = propensity_model.predict_proba(X_holdout)[:, 1]
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: FULL CLASSIFICATION METRICS SUITE (STANDING RULE, NOTEBOOK 50
#            SECTION 8) -- REAL, MEASURED ON THE REAL HOLDOUT POPULATION
# =============================================================================
_section("SECTION 7: Full Classification Metrics Suite -- Real Holdout Results")

_holdout_roc_auc = float(roc_auc_score(y_holdout, PROPENSITY_HOLDOUT))
_holdout_pr_auc = float(average_precision_score(y_holdout, PROPENSITY_HOLDOUT))
_holdout_log_loss = float(log_loss(y_holdout, PROPENSITY_HOLDOUT, labels=[0, 1]))

_fpr, _tpr, _roc_thresholds = roc_curve(y_holdout, PROPENSITY_HOLDOUT)
_pr_precision, _pr_recall, _pr_thresholds = precision_recall_curve(y_holdout, PROPENSITY_HOLDOUT)

# F1-optimal threshold on the real holdout PR curve -- same convention this
# platform's other notebooks use for reporting threshold-based metrics
# alongside threshold-free ones (honestly labeled as fit ON the holdout for
# reporting purposes, not as a claim of an independently-chosen operating
# point).
_f1_scores = np.where(
    (_pr_precision + _pr_recall) > 0,
    2 * _pr_precision * _pr_recall / np.where((_pr_precision + _pr_recall) > 0,
                                               _pr_precision + _pr_recall, 1.0),
    0.0,
)
_best_idx = int(np.argmax(_f1_scores))
_f1_threshold = float(_pr_thresholds[min(_best_idx, len(_pr_thresholds) - 1)])
_y_pred_at_threshold = (PROPENSITY_HOLDOUT >= _f1_threshold).astype(int)

_holdout_accuracy = float(accuracy_score(y_holdout, _y_pred_at_threshold))
_holdout_precision = float(precision_score(y_holdout, _y_pred_at_threshold, zero_division=0))
_holdout_recall = float(recall_score(y_holdout, _y_pred_at_threshold, zero_division=0))
_holdout_f1 = float(f1_score(y_holdout, _y_pred_at_threshold, zero_division=0))
_holdout_mcc = float(matthews_corrcoef(y_holdout, _y_pred_at_threshold))
_cm = confusion_matrix(y_holdout, _y_pred_at_threshold)
_tn, _fp, _fn, _tp = _cm.ravel()
_holdout_specificity = float(_tn / (_tn + _fp)) if (_tn + _fp) > 0 else 0.0

CLASSIFICATION_METRICS = {
    "roc_auc": _holdout_roc_auc,
    "pr_auc": _holdout_pr_auc,
    "log_loss": _holdout_log_loss,
    "f1_optimal_threshold": _f1_threshold,
    "accuracy": _holdout_accuracy,
    "precision": _holdout_precision,
    "recall": _holdout_recall,
    "f1": _holdout_f1,
    "specificity": _holdout_specificity,
    "matthews_corrcoef": _holdout_mcc,
    "confusion_matrix": {"tn": int(_tn), "fp": int(_fp), "fn": int(_fn), "tp": int(_tp)},
}
for _k, _v in CLASSIFICATION_METRICS.items():
    print(f"  {_k:>20}: {_v}")

MEETS_ROC_AUC_TARGET = _holdout_roc_auc >= MIN_ROC_AUC_TARGET
print(f"\nmin_propensity_model_roc_auc target ({MIN_ROC_AUC_TARGET}) met: {MEETS_ROC_AUC_TARGET} "
      f"(real, measured: {_holdout_roc_auc:.4f})")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: APPLY THE REAL TREATMENT-TIER POLICY (NOTEBOOK 50, SECTION 8) --
#            REAL PROPENSITY SCORE x REAL SEVERITY-SCORE MAGNITUDE
# =============================================================================
_section("SECTION 8: Apply the Real Treatment-Tier Policy")

# Both arrays below come from the SAME eligible-population row order
# (PROPENSITY_HOLDOUT from Section 6's predict_proba on X_holdout;
# HOLDOUT_ELIGIBLE_SEVERITY captured in Section 6 from the same source frame
# before it was freed) -- genuinely aligned row-for-row, not a re-scan of a
# differently-filtered population.
_median_propensity = float(np.median(PROPENSITY_HOLDOUT))
_median_severity = float(np.median(HOLDOUT_ELIGIBLE_SEVERITY)) if len(HOLDOUT_ELIGIBLE_SEVERITY) else 0.0

_tier_assignments = np.full(len(PROPENSITY_HOLDOUT), "Monitor", dtype=object)
_high_severity_mask = HOLDOUT_ELIGIBLE_SEVERITY >= _median_severity
_low_propensity_mask = PROPENSITY_HOLDOUT < _median_propensity
_tier_assignments[_low_propensity_mask & _high_severity_mask] = "Priority Outreach"
_tier_assignments[~_low_propensity_mask] = "Automated Nudge"

_tier_counts = {t: int((_tier_assignments == t).sum()) for t in ["Priority Outreach", "Automated Nudge", "Monitor"]}
print(f"Real median propensity-to-cure (HOLDOUT): {_median_propensity:.4f}")
print(f"Real median severity score (HOLDOUT)     : {_median_severity:.4f}")
print(f"Real treatment-tier population counts (HOLDOUT): {_tier_counts}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: CHARTS
# =============================================================================
_section("SECTION 9: Charts")

plt.figure(figsize=(7, 5))
plt.plot(_fpr, _tpr, label=f"ROC (AUC={_holdout_roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Problem 9 -- Propensity-to-Cure ROC Curve (Real Holdout)")
plt.legend()
plt.tight_layout()
_roc_chart_path = COLLECTIONS_CHARTS_DIR / "propensity_roc_curve_chart.png"
plt.savefig(_roc_chart_path, dpi=120)
plt.close()

_importances = propensity_model.feature_importances_
_top10_idx = np.argsort(_importances)[-10:]
plt.figure(figsize=(8, 6))
plt.barh([MONITORED_COLS[i] for i in _top10_idx], _importances[_top10_idx])
plt.xlabel("XGBoost Feature Importance")
plt.title("Problem 9 -- Top 10 Real Propensity-to-Cure Features")
plt.tight_layout()
_importance_chart_path = COLLECTIONS_CHARTS_DIR / "feature_importance_chart.png"
plt.savefig(_importance_chart_path, dpi=120)
plt.close()

print(f"Wrote: {_roc_chart_path}")
print(f"Wrote: {_importance_chart_path}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: PERSIST THE REAL TRAINED MODEL & WRITE MODELING RESULTS
# =============================================================================
_section("SECTION 10: Persist the Real Trained Model & Write Modeling Results")

_model_path = COLLECTIONS_MODELS_DIR / "collections_propensity_xgboost.joblib"
joblib.dump(propensity_model, _model_path)
print(f"Wrote: {_model_path}")

COLLECTIONS_MODELING_RESULTS = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "n_train_eligible_statements": int(X_train.shape[0]),
    "n_holdout_eligible_statements": int(X_holdout.shape[0]),
    "train_cure_rate": _train_cure_rate,
    "holdout_cure_rate": _holdout_cure_rate,
    "classification_metrics": CLASSIFICATION_METRICS,
    "min_propensity_model_roc_auc_target": MIN_ROC_AUC_TARGET,
    "meets_kpi_target": MEETS_ROC_AUC_TARGET,
    "treatment_tier_counts": _tier_counts,
    "median_propensity_holdout": _median_propensity,
    "median_severity_score_holdout": _median_severity,
    "model_path": str(_model_path),
    "train_seconds": _train_seconds,
    "warp_thread_count": WARP_THREAD_COUNT,
    "random_seed": RANDOM_SEED,
}
_results_path = COLLECTIONS_MODELING_DIR / "collections_modeling_results.json"
with open(_results_path, "w", encoding="utf-8") as f:
    json.dump(COLLECTIONS_MODELING_RESULTS, f, indent=2)
print(f"Wrote: {_results_path}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 11: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Modeling results file was written", _results_path.exists())
_all_checks_passed &= _check("Model artifact was persisted", _model_path.exists())
_all_checks_passed &= _check("ROC-AUC is a real, valid probability-ranking score in [0, 1]",
                              0.0 <= _holdout_roc_auc <= 1.0)
_all_checks_passed &= _check("PR-AUC is a real, valid score in [0, 1]", 0.0 <= _holdout_pr_auc <= 1.0)
_all_checks_passed &= _check("MCC is in the valid [-1, 1] range", -1.0 <= _holdout_mcc <= 1.0)
_all_checks_passed &= _check("Confusion matrix counts sum to the real holdout eligible population",
                              (_tn + _fp + _fn + _tp) == X_holdout.shape[0])
_all_checks_passed &= _check("Real TRAIN and HOLDOUT cure rates are both in (0, 1) -- neither degenerate",
                              0.0 < _train_cure_rate < 1.0 and 0.0 < _holdout_cure_rate < 1.0)
_all_checks_passed &= _check("Treatment-tier counts sum to the real holdout population",
                              sum(_tier_counts.values()) == len(PROPENSITY_HOLDOUT))
_all_checks_passed &= _check("Severity-score array used for tiering is row-aligned with the propensity "
                              "array (same length, same eligible-population source frame)",
                              len(HOLDOUT_ELIGIBLE_SEVERITY) == len(PROPENSITY_HOLDOUT))
_all_checks_passed &= _check("Reused Problem 8's real CUT_LOW/CUT_HIGH verbatim (no refit)",
                              CUT_LOW == P8_REUSE["cut_low"] and CUT_HIGH == P8_REUSE["cut_high"])
_all_checks_passed &= _check("WARP thread count matches Notebook 50's tightened cap exactly",
                              WARP_THREAD_COUNT == COLLECTIONS_POLICY["warp_resource_cap"]["warp_thread_count"])

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n\u2705 Section 11 complete -- all checks passed.")


# =============================================================================
# SECTION 12: WRITE NOTEBOOK 51 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 12: Write Notebook 51 Summary Artifact")

NB51_SUMMARY = {
    "notebook": "51_collections_optimization_modeling.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "results_path": str(_results_path),
    "model_path": str(_model_path),
    "holdout_roc_auc": _holdout_roc_auc,
    "meets_kpi_target": MEETS_ROC_AUC_TARGET,
    "train_cure_rate": _train_cure_rate,
    "holdout_cure_rate": _holdout_cure_rate,
    "warp_thread_count": WARP_THREAD_COUNT,
    "random_seed": RANDOM_SEED,
}
NB51_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_51_summary.json"
with open(NB51_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB51_SUMMARY, f, indent=2)
print(f"Wrote: {NB51_SUMMARY_PATH}")

_section("NOTEBOOK 51 COMPLETE")
print(f"Real HOLDOUT ROC-AUC (propensity-to-cure)     : {_holdout_roc_auc:.4f} "
      f"(target {MIN_ROC_AUC_TARGET}, met: {MEETS_ROC_AUC_TARGET})")
print(f"Real HOLDOUT PR-AUC                           : {_holdout_pr_auc:.4f}")
print(f"Real TRAIN / HOLDOUT cure rate                : {_train_cure_rate:.4f} / {_holdout_cure_rate:.4f}")
print(f"Real treatment-tier counts (HOLDOUT)          : {_tier_counts}")
print(f"Trained in {_train_seconds:.1f}s using {WARP_THREAD_COUNT} WARP-capped threads")
print(f"Model persisted to: {_model_path}")
print(
    "\nNext: 52_collections_optimization_validation_deployment.ipynb -- independent zero-randomness "
    "reproduction of this pipeline, bootstrap CIs on the real ROC-AUC/PR-AUC, and a real FastAPI "
    "collections-scoring service (with API-key auth and per-request explainability from day one, per "
    "this platform's now-standing hardening convention) proven via a live self-test."
)
